# BayCE-Net — Bayesian Channel Estimation for UAC-OFDM
**دفتر التدريب المتكامل** | كل خلية مستقلة، شغّلها بالترتيب (Shift+Enter)

يعرض تقدّم كل epoch مباشرة مع شريط تقدّم و NMSE على مجموعة التحقّق.

> تأكّد أن النواة أعلى اليمين هي **Python (paper2)**

## 1. الإعدادات والاستيراد

In [ ]:
import os, numpy as np, torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

DEV = "cuda" if torch.cuda.is_available() else "cpu"
print("PyTorch:", torch.__version__, "| device:", DEV)
if DEV == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
torch.manual_seed(0); np.random.seed(0)

# مجلد العمل — عدّله إن لزم
WORK = r"E:\PhDpapers\PAPER 2\code"
os.chdir(WORK)
print("working dir:", os.getcwd())
print("files:", [f for f in os.listdir() if f.endswith(('.py','.npz'))])

## 2. توليد البيانات
يبني مجموعة UAC-OFDM مع ضوضاء نبضية وخريطة عدم يقين. شغّلها مرّة؛ لو `bayce_dataset.npz` موجود تُتخطّى تلقائياً.

In [ ]:
DATA = "bayce_dataset.npz"
if os.path.exists(DATA):
    print(DATA, "موجود — تخطّي التوليد.")
else:
    import runpy
    runpy.run_path("generate_dataset.py")   # يشغّل السكربت كما هو
    print("تم توليد", DATA)

## 3. محمّل البيانات (تقسيم حسب القناة — لا تسرّب)

In [ ]:
class CEDataset(Dataset):
    def __init__(self, npz_path, split="train", train_frac=.7, val_frac=.1, seed=0):
        d = np.load(npz_path, allow_pickle=True)
        cid = d["chan_id"]; uniq = np.unique(cid)
        rng = np.random.default_rng(seed); rng.shuffle(uniq)
        n = len(uniq); ntr, nva = int(train_frac*n), int(val_frac*n)
        sets = {"train":uniq[:ntr], "val":uniq[ntr:ntr+nva], "test":uniq[ntr+nva:]}
        m = np.isin(cid, sets[split])
        Hls, s2, H = d["H_ls"][m], d["sigma2"][m], d["H_true"][m]
        self.x = np.stack([Hls.real, Hls.imag, np.log(s2+1e-8)],1).astype(np.float32)
        self.y = np.stack([H.real, H.imag],1).astype(np.float32)
        self.scale = np.clip(np.sqrt(np.mean(np.abs(Hls)**2,1,keepdims=True)),1e-6,None).astype(np.float32)
        print(f"[{split}] {m.sum()} samples from {len(sets[split])} channels")
    def __len__(self): return len(self.x)
    def __getitem__(self, i):
        s = self.scale[i]; x = self.x[i].copy(); x[:2] /= s
        return torch.from_numpy(x), torch.from_numpy(self.y[i]/s)

train_ds = CEDataset(DATA,"train"); val_ds = CEDataset(DATA,"val")
train_dl = DataLoader(train_ds, batch_size=64, shuffle=True, drop_last=True)
val_dl   = DataLoader(val_ds,   batch_size=64)

## 4. نموذج BayCE-Net (مراحل ثلاث + رأس بايزي مزدوج)

In [ ]:
class ImpulseGate(nn.Module):
    def __init__(self, ch=32):
        super().__init__()
        self.net = nn.Sequential(nn.Conv1d(3,ch,5,padding=2), nn.ReLU(),
                                 nn.Conv1d(ch,1,5,padding=2), nn.Sigmoid())
    def forward(self, x):
        w = self.net(x)
        gated = x[:,:2]*w; rest = x[:,2:]
        return torch.cat([gated,rest],1), w      # لا تعديل in-place

class SparseBackbone(nn.Module):
    def __init__(self, ch=64, n=5, p=0.1):
        super().__init__()
        self.inp = nn.Conv1d(3,ch,3,padding=1)
        self.blocks = nn.ModuleList([nn.Sequential(
            nn.Conv1d(ch,ch,3,padding=1), nn.ReLU(), nn.Dropout(p),
            nn.Conv1d(ch,ch,3,padding=1)) for _ in range(n)])
        self.act = nn.ReLU()
    def forward(self, x):
        h = self.inp(x)
        for b in self.blocks: h = self.act(h + b(h))
        return h

class BayCENet(nn.Module):
    def __init__(self, ch=64, n=5, p=0.1):
        super().__init__()
        self.gate = ImpulseGate(); self.backbone = SparseBackbone(ch,n,p)
        self.head_mu = nn.Conv1d(ch,2,3,padding=1)
        self.head_lv = nn.Conv1d(ch,1,3,padding=1)
    def forward(self, x):
        x,w = self.gate(x); f = self.backbone(x)
        mu = self.head_mu(f); lv = torch.clamp(self.head_lv(f),-10,10)
        return mu, lv, w

model = BayCENet().to(DEV)
print(f"BayCE-Net parameters: {sum(p.numel() for p in model.parameters()):,}")

## 5. دالة الخسارة البايزية (Gaussian NLL)

In [ ]:
def gaussian_nll(mu, log_var, y, l1=0.0):
    err2 = ((mu-y)**2).sum(1,keepdim=True)
    nll = 0.5*(torch.exp(-log_var)*err2 + log_var)
    return nll.mean() + l1*mu.abs().mean()

def mse_only(mu, log_var, y, l1=1e-4):
    return ((mu-y)**2).mean() + l1*mu.abs().mean()

## 6. التدريب — مع عرض تقدّم كل epoch
يستخدم `tqdm` لشريط تقدّم داخل كل epoch، ويطبع NMSE التحقّق بعد كل واحد. أول 8 epochs بـ MSE (warm-up)، ثم يتحوّل لـ NLL البايزي.

In [ ]:
from tqdm.notebook import tqdm

EPOCHS, WARMUP, LR, CLIP = 60, 8, 1e-3, 1.0
opt = torch.optim.Adam(model.parameters(), lr=LR)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, EPOCHS)

best, hist = 1e9, []
for ep in range(1, EPOCHS+1):
    use_nll = ep > WARMUP
    model.train(); tl = 0.0
    bar = tqdm(train_dl, desc=f"Epoch {ep:2d} [{'NLL' if use_nll else 'MSE'}]", leave=False)
    for x,y in bar:
        x,y = x.to(DEV), y.to(DEV)
        opt.zero_grad()
        mu,lv,_ = model(x)
        loss = gaussian_nll(mu,lv,y) if use_nll else mse_only(mu,lv,y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), CLIP)
        opt.step(); tl += loss.item()
        bar.set_postfix(loss=f"{loss.item():.4f}")
    sched.step()

    # NMSE على التحقّق
    model.eval(); num=den=0.0
    with torch.no_grad():
        for x,y in val_dl:
            x,y = x.to(DEV), y.to(DEV); mu,_,_ = model(x)
            num += ((mu-y)**2).sum().item(); den += (y**2).sum().item()
    vnmse = 10*np.log10(num/den+1e-12); hist.append(vnmse)
    star = ""
    if vnmse < best:
        best = vnmse; torch.save(model.state_dict(), "bayce_best.pt"); star=" <- best, saved"
    print(f"Epoch {ep:2d} [{'NLL' if use_nll else 'MSE'}] "
          f"train {tl/len(train_dl):.4f} | val NMSE {vnmse:6.2f} dB{star}")

print(f"\nBest val NMSE = {best:.2f} dB  ->  bayce_best.pt")

## 7. منحنى التعلّم

In [ ]:
plt.figure(figsize=(9,4))
plt.plot(range(1,len(hist)+1), hist, marker='o', ms=3)
plt.axvline(WARMUP+0.5, color='r', ls='--', label='MSE -> NLL')
plt.xlabel('Epoch'); plt.ylabel('Validation NMSE (dB)')
plt.title('BayCE-Net training'); plt.grid(alpha=.3); plt.legend(); plt.show()